# Generalization and the Test Set

**Topic:** What Does Learning Actually Mean?

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import ipywidgets as widgets
from ipywidgets import IntSlider, FloatSlider, Dropdown, ToggleButtons, Output, HBox, VBox, Label
from IPython.display import display, clear_output
from scipy import stats
from sklearn.datasets import make_classification, make_regression, make_circles, make_moons
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, KFold, StratifiedKFold
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from tkh_utils import PALETTE, FONT, base_layout


---
## What you'll explore

By the end of this demo you will be able to:

- **Describe** why the test set must be held out until the very end and never used during development
- **Explain** how peeking at the test set during training leads to inflated performance estimates
- **Interpret** the gap between a repeatedly-queried score and performance on truly unseen data as a direct measure of test-set overfitting

> **Tip:** Start with the **Small public set** and drag the **Submissions** slider upward. Watch the best public score climb higher and higher while the private score — the same model measured on data the leaderboard never saw — stays flat. The growing gap between the two lines is overfitting to the test set, built entirely out of luck. Then switch to the **Large public set** and notice the gap barely opens.

---
## How we got here

This is the capstone notebook for the ml_concepts/ folder. Every concept we explored leads here:

- **01_what_is_machine_learning.ipynb**: Learning means generalizing, not memorizing
- **04_bias_variance_tradeoff.ipynb**: Bias and variance both hurt generalization
- **05_overfitting_and_underfitting.ipynb**: Overfitting is failure to generalize
- **06_cross_validation.ipynb**: Cross-validation gives a reliable generalization estimate
- **07_loss_functions.ipynb** through **08_gradient_descent.ipynb**: The training machinery
- **09_the_no_free_lunch_theorem.ipynb** through **14_parametric_vs_nonparametric_models.ipynb**: The choices that determine how well a model generalizes

The test set is where we answer the only question that ultimately matters: **does the model work on data it has never seen?**

---
## Why this matters for data science

The test set is the only honest measure of model performance. Every other number — training accuracy, validation accuracy, cross-validation score — is an estimate used to make modeling decisions. Those decisions introduce a subtle bias: you have been optimizing your choices toward the validation data.

The test set corrects for that bias. It is data that influenced absolutely none of your decisions. Its score is the number you report to stakeholders and the number that predicts real-world performance.

---
## Try it yourself

In [2]:
from sklearn.model_selection import train_test_split
from IPython.display import HTML

# Data + candidate precompute — run once
np.random.seed(0)
X, y = make_classification(n_samples=3000, n_features=20, n_informative=5,
    n_redundant=2, flip_y=0.25, class_sep=0.7, random_state=0)
X_dev, X_pool, y_dev, y_pool = train_test_split(X, y, test_size=0.6, random_state=0)

rng = np.random.RandomState(1)
K = 300
pool_preds = []
for k in range(K):
    idx = rng.choice(len(X_dev), len(X_dev), replace=True)
    clf = DecisionTreeClassifier(
        max_depth=4, random_state=rng.randint(1_000_000)
    ).fit(X_dev[idx], y_dev[idx])
    pool_preds.append(clf.predict(X_pool))
correct = (np.array(pool_preds) == y_pool)   # (K, n_pool) boolean

PUBLIC_SIZES = {"Small public set (80)": 80, "Large public set (600)": 600}
_cache = {}

def get_climb(public_size):
    if public_size in _cache:
        return _cache[public_size]
    perm = np.random.RandomState(7).permutation(len(y_pool))
    pub_idx, prv_idx = perm[:public_size], perm[public_size:]
    pub  = correct[:, pub_idx].mean(1)
    prv  = correct[:, prv_idx].mean(1)
    best_pub, paired_prv = [], []
    bv, bi = -1.0, -1
    for k in range(K):
        if pub[k] > bv:
            bv, bi = pub[k], k
        best_pub.append(pub[bi])
        paired_prv.append(prv[bi])
    _cache[public_size] = (np.array(best_pub), np.array(paired_prv), pub)
    return _cache[public_size]

ACCENT = PALETTE.get("accent", PALETTE["secondary"])

out = Output()
_initialized = False

subs_slider = IntSlider(value=30, min=1, max=300, step=1,
    description="Submissions:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="460px"))
public_toggle = ToggleButtons(
    options=list(PUBLIC_SIZES.keys()),
    value="Small public set (80)",
    layout=widgets.Layout(width="460px"))

def render(n_subs, public_label):
    global _initialized
    best_pub, paired_prv, pub = get_climb(PUBLIC_SIZES[public_label])
    xs = list(range(1, n_subs + 1))

    # Order matters for fill="tonexty": private line first, then public line fills toward it
    traces = [
        go.Scatter(x=xs, y=pub[:n_subs].tolist(), mode="markers",
            marker=dict(color=PALETTE["muted"], size=4, opacity=0.35),
            name="Each submission (public)"),
        go.Scatter(x=xs, y=paired_prv[:n_subs].tolist(), mode="lines",
            line=dict(color=PALETTE["primary"], width=2.5),
            name="That model on hidden private set"),
        go.Scatter(x=xs, y=best_pub[:n_subs].tolist(), mode="lines",
            line=dict(color=PALETTE["secondary"], width=2.5),
            fill="tonexty", fillcolor="rgba(220,53,69,0.12)",
            name="Best public score kept"),
        go.Scatter(x=[n_subs], y=[float(best_pub[n_subs - 1])], mode="markers",
            marker=dict(size=13, color=PALETTE["primary"],
                        line=dict(color="white", width=2)),
            showlegend=False, name="Current"),
    ]

    fig = go.Figure(data=traces)
    fig.update_layout(base_layout(
        title=(f"After {n_subs} submissions — "
               f"public {best_pub[n_subs-1]:.3f} vs private {paired_prv[n_subs-1]:.3f}"),
        xaxis_title="Number of submissions (times the test set was queried)",
        yaxis_title="Accuracy"))
    fig.update_xaxes(range=[1, 300])
    fig.update_yaxes(range=[0.5, 0.92])

    gap = float(best_pub[n_subs - 1] - paired_prv[n_subs - 1])

    line1 = (f"Best public score after {n_subs} submissions: {best_pub[n_subs-1]:.3f}. "
             f"The SAME model on the hidden private set: {paired_prv[n_subs-1]:.3f}. "
             f"The {gap:+.3f} gap is what peeking bought you — and it vanishes on data "
             f"the leaderboard never saw.")

    line2 = ("With few submissions the gap is small: you have barely queried the test set."
             if n_subs <= 5 else
             "The private line stays flat near true skill while the public line keeps "
             "climbing. You are not building better models — you are finding ones that "
             "happen to fit this particular public slice.")

    line3 = ("A small public set is noisy, so the best-of-many public score is mostly luck. "
             "This is why a frequently-queried test set gets overfit."
             if "Small" in public_label else
             "A larger public set averages out the noise: the gap barely opens. Big, "
             "rarely-queried test sets resist this — the lesson of the notebook.")

    interp = (
        f'<div style="font-family:{FONT["family"]};font-size:13px;background:#f8f9fa;'
        f'padding:14px 18px;border-radius:6px;margin-top:8px;line-height:1.65;color:#333;">'
        f'<p style="margin:0 0 6px 0;">{line1}</p>'
        f'<p style="margin:0 0 6px 0;">{line2}</p>'
        f'<p style="margin:0;">{line3}</p>'
        f'</div>'
    )

    with out:
        clear_output(wait=True)
        fig.show()
        display(HTML(interp))
    _initialized = True

def on_change(change):
    render(subs_slider.value, public_toggle.value)

subs_slider.observe(on_change, names="value")
public_toggle.observe(on_change, names="value")

display(VBox([subs_slider, public_toggle, out]))
render(30, "Small public set (80)")

---
## What's happening?

Think of a driving test. You study the manual (training data), practice on familiar roads (validation set), and then take the real test on roads you have never driven before (test set). The practice score helps you improve. The test score is the only honest measure of whether you can actually drive.

The three splits serve three distinct purposes:
- **Training set:** What the model learns from
- **Validation set:** What you use to make decisions (hyperparameters, model selection, early stopping)
- **Test set:** What you evaluate on exactly once, after all decisions are final

Each time you check a model against the test set and use that score to decide what to try next, you let the test set influence your choices. Do it once and the damage is negligible. Do it hundreds of times — as in a Kaggle competition with a public leaderboard, or a team running experiment after experiment against the same held-out data — and something subtle takes over. Among many models of roughly equal real skill, the one that scores highest on a small test slice is usually just the luckiest, not the best. Keep selecting the current public leader and the public score climbs steadily, but that climb is borrowed from noise: on a truly hidden slice, those same models perform no better than average.

The widget shows exactly this. The best public score rises with every submission while the private score stays flat, and the gap between them is performance that exists only on the data you peeked at. A larger public set averages out the noise, which is why the gap shrinks when you switch to it — the practical lesson being to keep your test set large and query it rarely.

| Dataset split | Purpose | When to use it | What it tells you |
|---|---|---|---|
| Training | Model learning | Every training iteration | Training error (optimistic) |
| Validation | Model selection | After each experiment | Estimated generalization |
| Test | Final evaluation | Once, after all decisions are final | True generalization performance |

---
## Real-world example: Competition leaderboard overfitting

In a Kaggle competition, participants submit predictions and see their public leaderboard score (based on ~30% of the test set). Teams optimize their models toward this score across thousands of submissions. Many teams' final standings on the private leaderboard (the other 70%) are dramatically worse than their public score — they overfit to the 30% they could see.

This is test set contamination at scale. The same thing happens in industry when data scientists run too many experiments against the same held-out test set. Hold it out, evaluate once, report honestly.

---
## Key takeaway

> **The test set must never influence any modeling decision — it exists solely to measure generalization after all choices are final, and seeing it early gives you an inflated performance number that will not hold in production.**

---
*Next up: the **supervised/** folder — you have built a complete conceptual foundation. Now it is time to meet the algorithms. Let's go.*